[<img src="../quantumsymmetry_logo.png" alt="QuantumSymmetry" width="450"/>](https://github.com/dariopicozzi/quantumsymmetry)

> **Note:** if you are running this notebook on Google Colab, the next cell will install `quantumsymmetry` and its dependencies:

In [ ]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip -q install quantumsymmetry

# The Bravyi-Kitaev mapping

All the symmetry-adapted encodings we have built so far have used the **Jordan-Wigner (JW) mapping** between fermionic and qubit operators. In the JW mapping the $j$-th qubit stores the occupancy of the $j$-th spin-orbital, and the fermionic creation and annihilation operators map to

$$
a^{\dagger}_j \;\rightarrow\; \tfrac12 (X_j - i Y_j)\, Z_{j-1} \cdots Z_0, \qquad
a_j \;\rightarrow\; \tfrac12 (X_j + i Y_j)\, Z_{j-1} \cdots Z_0.
$$

The trailing string of Pauli $Z$ operators implements the fermionic sign rule. It has length $O(n)$, where $n$ is the number of qubits, which means a single fermionic excitation in Jordan-Wigner has Pauli weight $O(n)$.

The **Bravyi-Kitaev (BK) mapping** is an alternative fermion-to-qubit encoding that distributes the parity bookkeeping more cleverly between qubits. In BK, the operators $a^{\dagger}_j$, $a_j$ have a Pauli weight that scales as $O(\log n)$ rather than $O(n)$. BK is itself a Clifford basis change of JW that acts as an affine map on computational-basis bitstrings; this is exactly the kind of map that `quantumsymmetry` already uses to fold in Boolean symmetries.

On `Encoding` we can switch to the Bravyi-Kitaev mapping by passing `bravyi_kitaev=True`. The resulting **SAE-CAS-BK** encoding is unitarily equivalent to SAE-CAS: it uses the same number of qubits, has the same number of variational parameters, and yields the same eigenspectrum, but the per-term Pauli weights and entangling-gate counts of the resulting circuits are different.

1. [BeH₂: SAE-CAS vs SAE-CAS-BK](#BeH2)
2. [Pauli weight comparison](#weight)
3. [Unitary equivalence of the two encodings](#equivalence)


<a name="BeH2"></a>
## BeH₂: SAE-CAS vs SAE-CAS-BK

We re-use the BeH₂ CAS(2,3) example from [06_complete_active_space](06_complete_active_space.ipynb), but this time we ask for the Bravyi-Kitaev mapping:

In [ ]:
import quantumsymmetry

encoding_jw = quantumsymmetry.Encoding(
    atom = 'Be 0 0 0; H 0 0 1.33; H 0 0 -1.33',
    basis = 'sto-3g',
    CAS = (2, 3),
)
encoding_bk = quantumsymmetry.Encoding(
    atom = 'Be 0 0 0; H 0 0 1.33; H 0 0 -1.33',
    basis = 'sto-3g',
    CAS = (2, 3),
    bravyi_kitaev = True,
)

print('SAE-CAS     Hamiltonian:', encoding_jw.hamiltonian)
print()
print('SAE-CAS-BK  Hamiltonian:', encoding_bk.hamiltonian)

Both Hamiltonians have the same number of Pauli terms on the same number of qubits (2 qubits, 10 terms): the BK basis change is unitary, so it does not change the number of independent operators or the spectrum. What it does change is **which** Pauli operators appear: where SAE-CAS has a $Z_0 Z_1$ or $X_0 X_1$, SAE-CAS-BK might have a single-qubit $Z_1$ or a rearranged combination.

<a name="weight"></a>
## Pauli weight comparison

On systems with just a few qubits the Pauli weight is necessarily small, so to make the locality difference visible we look at a larger system. Consider the LiH molecule in the cc-pVDZ basis. This has 20 spin-orbitals, so the Jordan-Wigner string in the operator $a^{\dagger}_{19}$ would carry 19 $Z$ operators; in BK it carries roughly $\log_2 20 \approx 5$ of them.

We can confirm this by encoding a single creation operator on the highest-index spin-orbital under each mapping. We use tutorial [04_fermionic_operators](04_fermionic_operators.ipynb)'s `apply` method, but we disable the symmetry filtering by creating an encoding with no symmetries to remove (so that `apply` returns the bare JW or BK qubit operator):

In [ ]:
from openfermion import FermionOperator, count_qubits

atom_lih = 'Li 0 0 0; H 0 0 1.5949'
basis = 'cc-pvdz'

# A single creation operator on a high-index spin-orbital.
# This is *not* symmetric, so we deliberately disable symmetry
# reduction by setting symmetry=False; the encoding then just
# applies the chosen fermion-to-qubit mapping.
fop = FermionOperator('19^')

enc_jw = quantumsymmetry.Encoding(
    atom = atom_lih, basis = basis, symmetry = False,
)
enc_bk = quantumsymmetry.Encoding(
    atom = atom_lih, basis = basis, symmetry = False,
    bravyi_kitaev = True,
)

qop_jw = enc_jw.apply(fop)
qop_bk = enc_bk.apply(fop)

def max_pauli_weight(qop):
    return max(len(term) for term, _ in qop.terms.items()) if qop.terms else 0

print(f'JW: max Pauli weight of a_19^ = {max_pauli_weight(qop_jw)}')
print(f'BK: max Pauli weight of a_19^ = {max_pauli_weight(qop_bk)}')

The BK weight is dramatically smaller — and this difference carries over into shallower circuits when we Trotterise an exponential of a sum of fermionic operators, as in UCCSD. The trade-off is that in BK each individual qubit no longer has a direct "this spin-orbital is occupied" interpretation, since occupancies are stored in linear combinations of qubits.

<a name="equivalence"></a>
## Unitary equivalence of the two encodings

The SAE-CAS and SAE-CAS-BK Hamiltonians are unitarily equivalent: their eigenspectra agree exactly. We can verify this for our BeH₂ example by diagonalising both Hamiltonians as dense matrices:

In [ ]:
import numpy as np
from openfermion.linalg import qubit_operator_sparse

H_jw = qubit_operator_sparse(encoding_jw.hamiltonian).toarray()
H_bk = qubit_operator_sparse(encoding_bk.hamiltonian).toarray()

eigs_jw = np.sort(np.linalg.eigvalsh(H_jw))
eigs_bk = np.sort(np.linalg.eigvalsh(H_bk))

print('SAE-CAS     spectrum:', np.round(eigs_jw, 8))
print('SAE-CAS-BK  spectrum:', np.round(eigs_bk, 8))
print('Max abs difference  :', float(np.max(np.abs(eigs_jw - eigs_bk))))

The two spectra coincide to numerical precision, confirming that SAE-CAS-BK is just a relabelling (a Clifford basis change) of SAE-CAS, with no impact on what the Hamiltonian *says* and potentially a real impact on how cheaply the corresponding circuits run on hardware.

<p style="text-align: left"> <a href="06_complete_active_space.ipynb" />< Previous: The complete active space (CAS) approximation</a> </p>
<p style="text-align: right"> <a href="08_periodic.ipynb" />Next: Periodic systems and crystals ></a> </p>